# Completed three-task results

Read saved metrics, confidence intervals, corrected tests and plots after the full
server check succeeds. Models and bootstrap inference are not rerun; the verifier checks saved metric/test consistency.
Set `IIVV_RESULTS` to the copied complete output directory.


In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

REPO = Path.cwd()
if not (REPO / "run_revision.py").is_file():
    REPO = REPO.parent
assert (
    REPO / "run_revision.py"
).is_file(), "Start Jupyter in the repository or manuscript directory"
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from revision.data import TASK_PROTOCOL, TASK_TITLES, TASK3_CAPTION


def csv(path, limit=None):
    path = Path(path)
    if not path.is_file():
        display(Markdown(f"Pending/unavailable: `{path.name}`"))
        return pd.DataFrame()
    try:
        table = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        table = pd.DataFrame()
    display(table if limit is None else table.head(limit))
    return table


def images(paths):
    for path in paths:
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))
        if "cancer_vs_lr_hr" in str(path):
            display(Markdown(TASK3_CAPTION.replace("\n", " ")))


ROOT = REPO / os.environ.get("IIVV_RESULTS", "results/revision-three-tasks-v3")
manifest = json.loads((ROOT / "run_manifest.json").read_text())
assert (
    manifest["config"].get("task_protocol") == TASK_PROTOCOL
), "Old runs cannot be relabeled"
assert not manifest.get("synthetic"), "Synthetic checks are not clinical results"
from revision.verification import verify_results

display(verify_results(ROOT))
TASKS = manifest["config"]["tasks"]

## AUC, accuracy, F1, precision, SN, SP, PPV, NPV and coverage

In [ ]:
metrics = csv(ROOT / "classification_outputs.csv")
summary = csv(ROOT / "all_metrics.csv")
selected = summary[
    summary.stream.isin(
        ["auc_selected", "acc_selected", "f1_selected", "sensitivity_selected"]
    )
]
columns = [
    c
    for c in [
        "task",
        "stream",
        "evaluation",
        "unit",
        "auc",
        "accuracy",
        "balanced_accuracy",
        "f1",
        "precision",
        "sensitivity",
        "specificity",
        "ppv",
        "npv",
        "coverage",
    ]
    if c in selected
]
display(selected[columns])
csv(ROOT / "unavailable_results.csv")

## Best training-selected combinations in each outer fold

A separate configuration is selected per fold/objective/model. Do not choose one
universal configuration using outer-test scores. Inner scores are tuning evidence.


In [ ]:
parameters = pd.read_csv(ROOT / "review_selected_parameters.csv")
expanded = pd.json_normalize(parameters.candidate.map(json.loads)).add_prefix("chosen_")
display(pd.concat([parameters.drop(columns="candidate"), expanded], axis=1))
TASK = "healthy_vs_cancer"
OBJECTIVE = "auc"
MODEL = "svm"
display(
    parameters[
        (parameters.task == TASK) & (parameters.stream == f"{OBJECTIVE}_{MODEL}")
    ]
)
search = pd.read_csv(ROOT / TASK / "inner_search.csv")
display(
    search[search.model == MODEL]
    .sort_values(f"objective_{OBJECTIVE}", ascending=False)
    .head(25)
)
display(FileLink(os.path.relpath(ROOT / "parameter-explorer.html", Path.cwd())))

## ROC curves and confusion matrices

Change STREAM to `auc_svm`, `f1_selected`, `sensitivity_selected`, etc.
Task 3 Low-grade dysplasia includes normal/benign sites and mild dysplasia;
High-grade dysplasia includes moderate/severe dysplasia and CIS; healthy sites are excluded.


In [ ]:
STREAM = "auc_selected"
for task in TASKS:
    display(Markdown(f"### {TASK_TITLES[task]}"))
    for folder in sorted(ROOT.glob(f"plots_{task}_*/{STREAM}")):
        images(
            [
                p
                for p in [folder / "ROC.png", folder / "CM.png", folder / "CM_norm.png"]
                if p.is_file()
            ]
        )
    csv(ROOT / task / "per_class_metrics.csv")
    for path in sorted((ROOT / task).glob(f"nested_cv_{STREAM}*confusion*.csv")):
        csv(path)

## Confidence intervals, corrected CV statistics and clinical operating points

Intervals cluster whole patients and condition on frozen predictions. Corrected
resampled tests approximate dependence between folds. Holm correction includes
all scheduled unique hypotheses, including unavailable comparisons. PPV equals precision.


In [ ]:
csv(ROOT / "corrected_resampled_tests_holm.csv")
for task in TASKS:
    for name in [
        "cv_metrics_summary.csv",
        "outer_fold_metrics.csv",
        "outer_fold_per_class_metrics.csv",
        "location_stratified_metrics.csv",
    ]:
        csv(ROOT / task / name, 40)
    for path in sorted((ROOT / task).glob("nested_cv_*selected*operating*.csv")):
        csv(path)

## Covariate controls, demographic splits and repeatability

In [ ]:
for task in TASKS:
    for name in [
        "matched_covariate_control.json",
        "cv_demographics_notes.json",
        "classes.json",
    ]:
        path = ROOT / task / name
        if path.is_file():
            display(json.loads(path.read_text()))
    for name in [
        "cv_population_summary.csv",
        "cv_pathology_breakdown.csv",
        "cv_demographics_availability.csv",
        "ranking_stability.csv",
    ]:
        csv(ROOT / task / name, 40)

## Selected coordinate identities, input sizes and measured performance

In [ ]:
csv(ROOT / "final_search_budget.csv")
for task in TASKS:
    for name in ["measured_computation_times.csv", "hardware_requirements.csv"]:
        csv(ROOT / task / name, 40)
    path = ROOT / task / "frequency_and_pattern_identities.json"
    display(FileLink(os.path.relpath(path, Path.cwd())))
    failures = json.loads((ROOT / task / "candidate_failures.json").read_text())
    display({"task": task, "rejected_candidates": len(failures)})
    display(pd.DataFrame(failures).head(20))

## Calibration, Brier scores, decision curves and prevalence assumptions

These outputs use outer-training calibration and operating thresholds. Decision
curves and assumed prevalences are retrospective summaries, not validated referral
rules. Intervals remain conditional on frozen outer predictions.


In [ ]:
for task in TASKS:
    display(Markdown(f"### {TASK_TITLES[task]}"))
    for suffix in [
        "calibration.csv",
        "decision_curve.csv",
        "prevalence_projections.csv",
    ]:
        csv(ROOT / task / f"nested_cv_{STREAM}_{suffix}", 30)
    for unit in ["site", "patient_weighted_site", "patient"]:
        csv(ROOT / task / f"nested_cv_{STREAM}_{unit}_confidence_intervals.csv")
        csv(
            ROOT
            / task
            / f"nested_cv_{STREAM}_{unit}_per_class_confidence_intervals.csv",
            40,
        )
    for folder in sorted(ROOT.glob(f"plots_{task}_*/{STREAM}")):
        images(
            [
                p
                for p in [folder / "calibration.png", folder / "decision_curve.png"]
                if p.is_file()
            ]
        )
    available = pd.read_csv(ROOT / task / "metrics_summary.csv")
    display(
        available[
            [
                c
                for c in ["stream", "unit", "brier", "log_loss", "coverage"]
                if c in available
            ]
        ]
    )

## Retained ablations and matched clinical controls

These compare only retained alternatives. Excluded selectors/representations are
not new negative findings. Missing covariates and failed model streams stay visible.


In [ ]:
for task in TASKS:
    available = pd.read_csv(ROOT / task / "metrics_summary.csv")
    display(Markdown(f"### {TASK_TITLES[task]}"))
    display(available[available.stream.str.startswith("ablation_")])
    display(
        available[
            available.stream.str.contains("covariate|combined|matched", regex=True)
        ]
    )